# 00 - Setup del entorno y descarga de datos

**Responsable principal:** parte general/compartida (todo el equipo debe correrla igual, no es un TODO individual como los notebooks 01-05).

**Objetivo:** dejar `data/raw/` con la misma muestra de datos en la maquina de cualquier integrante, para que los notebooks 01-05 den resultados reproducibles.

## Por que una muestra y no el dataset completo

El dataset completo de la competencia son **158 GB** (68 archivos de `train_landmarks/*.parquet`, ~1.5 GB cada uno). Para un EDA no hace falta: `train.csv` ya trae metadata de las 67,208 secuencias completas (participante, frase, archivo), y un par de archivos de landmarks alcanzan de sobra para describir estructura, valores faltantes y visualizar secuencias.

Por eso descargamos, con la CLI de `kaggle` (no `kagglehub`: para archivos individuales devuelve el contenido comprimido en zip pero nombrado sin `.zip`, lo cual rompe `pd.read_csv`/`pd.read_parquet` en silencio):
- `train.csv`, `supplemental_metadata.csv`, `character_to_prediction_index.json` (completos, pesan pocos MB)
- Una **muestra fija** de 2 archivos de `train_landmarks/` (`config.SAMPLE_LANDMARK_PATHS`) -- los mismos para todo el equipo.

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path
from src import config


## 1. Credenciales de Kaggle

Requiere `~/.kaggle/kaggle.json` (API token de https://www.kaggle.com/settings) y haber aceptado las reglas de la competencia: https://www.kaggle.com/competitions/asl-fingerspelling/rules

In [ ]:
!pip install -q kaggle


## 2. Descargar metadata completa (train.csv y compania)

Estos archivos vienen comprimidos como `<nombre>.zip` aunque se pida solo uno; `unzip -o` los deja listos y `rm` limpia el zip.

In [ ]:
config.DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

!kaggle competitions download -c asl-fingerspelling -f train.csv -p {config.DATA_RAW_DIR}
!kaggle competitions download -c asl-fingerspelling -f supplemental_metadata.csv -p {config.DATA_RAW_DIR}
!kaggle competitions download -c asl-fingerspelling -f character_to_prediction_index.json -p {config.DATA_RAW_DIR}


In [ ]:
!cd {config.DATA_RAW_DIR} && unzip -o train.csv.zip && unzip -o supplemental_metadata.csv.zip
!cd {config.DATA_RAW_DIR} && rm -f train.csv.zip supplemental_metadata.csv.zip


## 3. Descargar la muestra fija de landmarks

`config.SAMPLE_LANDMARK_PATHS` es la lista oficial -- si necesitas mas cobertura para tu parte del EDA, agrega archivos ahi (no descargues otros por tu cuenta) para que todo el equipo siga viendo los mismos datos.

In [ ]:
landmarks_dir = config.train_landmarks_dir()
landmarks_dir.mkdir(parents=True, exist_ok=True)

for rel_path in config.SAMPLE_LANDMARK_PATHS:
    fname = Path(rel_path).name
    if (landmarks_dir / fname).exists():
        print("ya existe:", fname)
        continue
    !kaggle competitions download -c asl-fingerspelling -f {rel_path} -p {landmarks_dir}
    !cd {landmarks_dir} && unzip -o {fname}.zip && rm -f {fname}.zip


## 4. Verificar el contenido descargado

In [ ]:
for p in sorted(config.DATA_RAW_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(config.DATA_RAW_DIR), f"({p.stat().st_size / 1e6:.1f} MB)")


## 5. Probar la carga con `src/data_loading.py`

Si esto corre sin errores, el resto de notebooks (01-05) ya pueden usar `dl.load_train_index()` / `dl.load_landmarks(dl.landmark_path(...))` sin configuracion adicional.

In [ ]:
from src import data_loading as dl

train_df = dl.load_train_index()
print(train_df.shape)

sample_landmarks = dl.load_landmarks(dl.landmark_path(config.SAMPLE_LANDMARK_PATHS[0]))
sample_landmarks.shape


## Notas

- Si en algun momento se necesita el dataset completo (no para este EDA), es `kagglehub.competition_download("asl-fingerspelling")` -- descarga los 158 GB en un solo archivo no reanudable, así que conviene tener espacio y tiempo de sobra.